### TL;DR

In order to use auto calibration, do the following:

1. Call `EDSespm::plot_table` to see how shifted the dataset is.
2. Call `EDSespm::auto_calibrate` with `window` you obtained in step 1.
   This does the following:
   -  Afflinely correct the energy axis of the dataset
   -  Correct the line table with higher degree polynomial
3. Call `EDSespm::build_G` with `use_calibration=True` if you want to use the higher degree correction.

In [ ]:
%matplotlib qt
# %matplotlib widget

import json

import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

from espm.conf import SYMBOLS_PERIODIC_TABLE
from espm.datasets.eds_spim import gaussian_with_bg

In [ ]:
FILENAME = "../playground/X3-13MAY22_MAP06.bcf"  # Change this to your dataset
BIN = 32  # Scale for rebinning

WINDOW = 0.05
FILTER_CS = 0.1  # Keep lines with cs >= FILTER_CS * CS_max
FILTER_CONC = 1.0  # Keep elements with concentration >= FILTER_CONC %
DEGREE = 2  # The degree of polynomial used to calibrate the energy axis

MAX_DEGREE = 10

Matplotlib setup, please ignore.

In [ ]:
with open(SYMBOLS_PERIODIC_TABLE, "r") as f:
    SPT = json.load(f)["table"]

cmap = plt.get_cmap("tab10")

In [ ]:
def plot_table(ax, table, elements, energy_axis, l1="--", l2="-.", bell=False):
    xs = []
    colours = []
    linestyles = []

    legend = {}

    bell_segments = []
    bell_colours = []

    hover_data = []

    idx = set()

    for i, elt in enumerate(elements):
        elt_num_str = str(SPT[elt]["number"])
        if elt_num_str not in table:
            continue
        db_entries = table[elt_num_str]

        colour = cmap(i)
        legend[elt] = colour

        for name, data in db_entries.items():
            energy = data["energy"]

            xs.append(energy)
            colours.append(colour)
            linestyles.append(l1 if energy not in idx else l2)

            hover_data.append({"x": energy, "name": f"{elt} {name}", "color": colour})

            if bell and "amplitude" in data:
                sigma = data["sigma"]
                A = data["amplitude"]
                C = data["bg"]
                # m = data["background_slope"]

                mask = (energy_axis > energy - 3 * sigma) & (
                    energy_axis < energy + 3 * sigma
                )
                x_axis = energy_axis[mask]

                y_gauss = gaussian_with_bg(x_axis, A, energy, sigma, C)

                points = np.column_stack([x_axis, y_gauss])
                bell_segments.append(points)
                bell_colours.append(colour)

            idx.add(energy)

    vline_collection = ax.vlines(
        x=xs,
        ymin=-0.1,
        ymax=2 * BIN * BIN,
        colors=colours,
        linestyles=linestyles,
        linewidths=1,
        pickradius=5,
    )

    if bell:
        bell_collection = LineCollection(
            bell_segments, colors=bell_colours, linewidths=1.5
        )
        ax.add_collection(bell_collection)

    for elt, col in legend.items():
        ax.plot([], [], color=col, label=elt, lw=1)
    ax.legend()

    annot = ax.annotate(
        "",
        xy=(0, 0),
        xytext=(10, 10),
        textcoords="offset points",
        bbox={"boxstyle": "round", "fc": "w", "alpha": 0.9, "ec": "gray"},
    )
    annot.set_visible(False)

    def on_hover(event):
        if event.inaxes == ax:
            contained, info = vline_collection.contains(event)

            if contained:
                line_idx = info["ind"][0]
                item = hover_data[line_idx]

                annot.xy = (item["x"], event.ydata)
                annot.set_text(f"{item['name']}: {item['x']:.3f}")
                annot.get_bbox_patch().set_edgecolor(item["color"])

                if not annot.get_visible():
                    annot.set_visible(True)
                    fig.canvas.draw_idle()
                return

        if annot.get_visible():
            annot.set_visible(False)
            fig.canvas.draw_idle()

    fig = ax.figure
    fig.canvas.mpl_connect("motion_notify_event", on_hover)


def plot(funcs, title):
    fig, ax = plt.subplots()

    for func in funcs:
        func(ax)

    ax.set_title(title)
    ax.set_xlabel("Energy (keV)")
    ax.set_ylabel("Intensity (counts)")

    ax.legend()

    def on_press(event):
        if event.key == " ":
            toolbar = event.canvas.toolbar
            if toolbar.mode != "pan/zoom":
                toolbar.pan()

    def on_release(event):
        if event.key == " ":
            toolbar = event.canvas.toolbar
            if toolbar.mode == "pan/zoom":
                toolbar.pan()

    fig.canvas.mpl_connect("key_press_event", on_press)
    fig.canvas.mpl_connect("key_release_event", on_release)

    plt.show()


def plot_avg(ax, avg_spectrum, energy_axis, **kwargs):
    ax.plot(energy_axis, avg_spectrum, **kwargs)

### Load Dataset

In [ ]:
signals = hs.load(FILENAME)
signal = signals[1].rebin(scale=(BIN, BIN, 1)).isig[0.2:]
signal.set_signal_type("EDS_espm")

signal.set_analysis_parameters(
    thickness=10e-5,
    density=4.1,
    detector_type="SDD_efficiency.txt",
    width_slope=0.01,
    width_intercept=0.065,
    geom_eff=None,
    xray_db="200keV_xrays.json",
)
signal.change_dtype("float64")

In [ ]:
average_spectrum = signal.average_spectrum
energy_axis = signal.energy_axis
theoretical_table = signal.model.db_dict

elements = signal.metadata.Sample.elements
# elements.remove("Re")

elements

### Result You should Expect :)

In [ ]:
s = signal.deepcopy()
s.auto_calibrate(WINDOW)
s_filter = signal.deepcopy()
s_filter.auto_calibrate(WINDOW, filter_cs=FILTER_CS, filter_conc=FILTER_CONC)

plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, linewidth=3, label="Input"
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
    ]
    + [
        lambda ax: plot_avg(
            ax,
            s.average_spectrum,
            s.energy_axis,
            linewidth=3,
            label="Calibrated",
        ),
        lambda ax: plot_table(
            ax,
            s.model.calibrated_db_dict,
            elements,
            s.energy_axis,
            l1=":",
            l2="-",
        ),
    ]
    + [
        lambda ax: plot_avg(
            ax,
            s_filter.average_spectrum,
            s_filter.energy_axis,
            linewidth=3,
            label="Calibrated (Filtered)",
        ),
        lambda ax: plot_table(
            ax,
            s_filter.model.calibrated_db_dict,
            elements,
            s_filter.energy_axis,
            l1=":",
            l2="-",
        ),
    ],
    "Original vs Calibrated",
)

### Average Spectrum vs Theoretical Lines

In [ ]:
signal.plot_table()
plt.show()

### Independently and Naively Fitted Lines

Dashed lines (- - -) are theoretical lines, and dotted lines (. . . .) are fitted lines

In [ ]:
plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, color="k", linewidth=3, label="Input"
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            signal.fit_table(WINDOW, filter_cs=0.0, filter_conc=0.0),
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    "Independent Naive Line Fit",
)

### Independently and Fitted Lines with Cross Section Filter

Dashed lines (- - -) are theoretical lines, and dotted lines (. . . .) are fitted lines

> Only lines with cross section $\geq$ `FILTER_CS` $\cdot$ (max cs of the element) are kept

In [ ]:
plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, color="k", linewidth=3, label="Input"
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            signal.fit_table(WINDOW, filter_cs=FILTER_CS, filter_conc=0.0),
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    f"Independent Line Fit with CS Filter {FILTER_CS}",
)

### Independently and Fitted Lines with Cross Section and Concentration Filter

Dashed lines (- - -) are theoretical lines, and dotted lines (. . . .) are fitted lines

> Only lines with cross section $\geq$ `FILTER_CS` $\cdot$ (max cs of the element) are kept

> Only elements with concentration $\geq$ `FILTER_CONC` % are kept

In [ ]:
from espm.utils import quant_spectrum

quant_spectrum(signal.mean(axis=(0, 1)))[0]

In [ ]:
plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, color="k", linewidth=3, label="Input"
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            signal.fit_table(window=0.05, filter_cs=FILTER_CS, filter_conc=FILTER_CONC),
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    f"Independent Line Fit with CS Filter {FILTER_CS} and Concentration Filter {FILTER_CONC}",
)

### Lines Calibrated using (Unweighted) Polynomial Estimation [Bad]

Dashed lines (- - -) are theoretical lines, and dotted lines (. . . .) are fitted lines

> Every line is independently fitted and a polynomial is fitted with the original and calibrated position.

In [ ]:
s = signal.deepcopy()
s.auto_calibrate(
    WINDOW,
    degree=DEGREE,
    weighted=False,
)

plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, color="k", linewidth=3, label="Input"
        ),
        lambda ax: plot_avg(
            ax,
            s.average_spectrum,
            s.energy_axis,
            linewidth=3,
            label="Calibrated",
            color="r",
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            s.model.calibrated_db_dict,
            elements,
            s.energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    f"Unweighted Polynomial Fitting (degree {DEGREE})",
)

### Lines Calibrated using (Weighted) Polynomial Estimation [Good]

Dashed lines (- - -) are theoretical lines, and dotted lines (. . . .) are fitted lines

> Every line is independently fitted and a polynomial is fitted with the original and calibrated position, with amplitude of each peak as weight.

In [ ]:
s = signal.deepcopy()
s.auto_calibrate(
    WINDOW,
    degree=DEGREE,
)

plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, color="k", linewidth=3, label="Input"
        ),
        lambda ax: plot_avg(
            ax,
            s.average_spectrum,
            s.energy_axis,
            linewidth=3,
            label="Calibrated",
            color="r",
        ),
        lambda ax: plot_table(ax, theoretical_table, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            s.model.calibrated_db_dict,
            elements,
            s.energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    f"Weighted Polynomial Fitting (degree {DEGREE})",
)

### Comparing Polynomial of Different Degrees

We see that most low degree polynomial agree that the calibration should be approximately linear, and degree 2 polynomial behaves the best. (For a polynomial with too high degree, overfitting will happen, which is not good)

In [ ]:
def plot_polys(filter_cs, filter_conc):
    _, ax = plt.subplots()

    lines = []

    for d in range(1, MAX_DEGREE + 1):
        s = signal.deepcopy()
        _, y_poly, _ = s.auto_calibrate(
            degree=d, window=0.05, filter_cs=filter_cs, filter_conc=filter_conc
        )
        ax.plot(energy_axis, np.polyval(y_poly, energy_axis), label=f"degree {d}")

    ax.plot(
        energy_axis, energy_axis, label="identity", color="k", linewidth=3, zorder=50
    )

    x = []
    y = []
    c = []

    calibrated = signal.fit_table(
        window=0.05, filter_cs=filter_cs, filter_conc=filter_conc
    )

    for i, el in enumerate(calibrated):
        lines = calibrated[el]
        for line in lines.values():
            x.append(line["theoretical"])
            y.append(line["energy"])
            c.append(cmap(i))

    ax.scatter(x, y, c=c, zorder=100)

    ax.set_aspect("equal")
    ax.set_box_aspect(1)
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 20)
    ax.legend(loc="upper right")

    ax.set_title(f"Theoretical vs Calibrated Energy ({filter_cs}, {filter_conc})")
    ax.set_xlabel("Theoretical Energy (keV)")
    ax.set_ylabel("Calibrated Energy (keV)")

    plt.show()

#### Polynomial Fittings Without Filtering by Cross Section

In [ ]:
plot_polys(0.0, 0.0)

#### Polynomial Fittings With Filtering by Cross Section

In [ ]:
plot_polys(FILTER_CS, FILTER_CONC)

### Visualising the `G` Built by Different Calibration Methods

At the end, we will plot a graph with the MSE of each method. We will see that weighted poly fit with degree 2 is generally very good, and naive method is overfitted.

In [ ]:
from espm.estimators.smooth_nmf import SmoothNMF


def run_decomp(name, build_g_func):
    s = signal.deepcopy()
    build_g_func(s)

    estimator = SmoothNMF(
        n_components=3,
        G=s.G,
        max_iter=500,
        tol=1e-5,
        init="nndsvdar",
        random_state=42,
        hspy_comp=True,
    )

    s.decomposition(algorithm=estimator)

    G_est = estimator.G_
    W_est = estimator.W_
    H_est = estimator.H_

    reconstructed = G_est @ W_est @ H_est
    reconstructed_mean = reconstructed.mean(1)

    mse = np.mean((reconstructed_mean - s.average_spectrum) ** 2)
    mae = np.mean(np.abs(reconstructed_mean - s.average_spectrum))

    print(f"{name} MSE: {mse:.6f}, MAE: {mae:.6f}")
    return reconstructed_mean, mse, mae, G_est

In [ ]:
rec_uncal, mse_uncal, mae_uncal, G_uncal = run_decomp(
    "Uncalibrated", lambda s: s.build_G()
)

In [ ]:
def build_calibrated_naive(s):
    s.model.db_dict = s.fit_table(window=WINDOW, filter_cs=0.0, filter_conc=0.0)
    s.build_G()


rec_naive, mse_naive, mae_naive, G_naive = run_decomp(
    "peak fit", build_calibrated_naive
)

In [ ]:
rec_poly_unweighted = {}
mse_poly_unweighted = {}
mae_poly_unweighted = {}
G_poly_unweighted = {}


def build_poly_unweighted(s, d):
    s.auto_calibrate(
        WINDOW,
        degree=d,
        weighted=False,
    )
    s.build_G(use_calibration=True)


for d in range(1, MAX_DEGREE + 1):
    rec, mse, mae, G = run_decomp(
        f"Unweighted Poly Fit degree {d}", lambda s, d=d: build_poly_unweighted(s, d)
    )
    rec_poly_unweighted[d] = rec
    mse_poly_unweighted[d] = mse
    mae_poly_unweighted[d] = mae
    G_poly_unweighted[d] = G

In [ ]:
rec_poly, mse_poly, mae_poly, G_poly = {}, {}, {}, {}


def build_poly(s, d):
    s.auto_calibrate(WINDOW, degree=d)
    s.build_G(use_calibration=True)


for d in range(1, MAX_DEGREE + 1):
    rec, mse, mae, G = run_decomp(
        f"Weighted Poly Fit degree {d}", lambda s, d=d: build_poly(s, d)
    )
    rec_poly[d] = rec
    mse_poly[d] = mse
    mae_poly[d] = mae
    G_poly[d] = G

In [ ]:
s = signal.deepcopy()
s.auto_calibrate(WINDOW)
s_unweighted = signal.deepcopy()
s_unweighted.auto_calibrate(WINDOW, weighted=False)

plot(
    [
        lambda ax: plot_avg(
            ax, average_spectrum, energy_axis, linewidth=3, label="Input"
        ),
        lambda ax: plot_avg(
            ax,
            s.average_spectrum,
            s.energy_axis,
            linewidth=3,
            label="Calibrated",
        ),
        lambda ax: plot_avg(
            ax,
            rec_uncal,
            energy_axis,
            linewidth=3,
            label=f"Uncalibrated {mse_uncal:.3f}",
            linestyle="-.",
        ),
        lambda ax: plot_avg(
            ax,
            rec_naive,
            energy_axis,
            linewidth=3,
            label=f"Naive {mse_naive:.3f}",
            linestyle="-.",
        ),
    ]
    + [
        lambda ax, d=d: plot_avg(
            ax,
            rec_poly[d],
            s.energy_axis,
            label=f"Weighted Poly deg {d} {mse_poly[d]:.3f}",
            linewidth=3,
        )
        for d in range(2, 2 + 1)
    ]
    # + [
    #     lambda ax, d=d: plot_avg(
    #         ax,
    #         rec_poly_unweighted[d],
    #         s.energy_axis,
    #         label=f"Unweighted Poly deg {d} {mse_poly_unweighted[d]:.3f}",
    #         linestyle="--",
    #     )
    #     for d in range(1, MAX_DEGREE + 1)
    # ]
    + [
        lambda ax: plot_table(
            ax, theoretical_table, elements, energy_axis, l1="-.", l2="-."
        )
    ]
    + [
        lambda ax: plot_table(
            ax,
            s.model.calibrated_db_dict,
            elements,
            s.energy_axis,
            l1=":",
            l2="-",
            # bell=True,
        ),
    ],
    # + [
    #     lambda ax: plot_table(
    #         ax,
    #         s_unweighted.model.calibrated_db_dict,
    #         elements,
    #         s_unweighted.energy_axis,
    #         l1="--",
    #         l2="-.",
    #         # bell=True,
    #     ),
    # ],
    "G of Different Methods",
)